In [ ]:
import random
random.seed(4)

# Grand Adventure Reference

This reference uses fixed choices so the complete adventure can run without keyboard interaction.

## Milestone 1

### Meet your hero

The hero starts with a name, 20 health points, and an empty inventory. Methods change the hero's state.

In [ ]:
class Hero:
    def __init__(self, name):
        self.name = name
        self.health = 20
        self.inventory = []

    def take_damage(self, amount):
        self.health = self.health - amount
        return self.health

    def pick_up(self, item):
        self.inventory.append(item)

## Milestone 2

### Build the world

Three flat dictionaries hold descriptions, composite-key exits, and room items. The helpers describe a room and return either the destination or the unchanged room.

In [ ]:
descriptions = {"cave": "A damp cave.", "forest": "A sunny forest.", "river": "A rushing river."}
exits = {"cave east": "forest", "forest west": "cave", "forest east": "river"}
room_items = {"forest": "sword", "river": "shield"}


def describe(descriptions, room):
    return descriptions[room]


def move(exits, current, direction):
    if direction == "q":
        return "quit"

    move_key = current + " " + direction
    if move_key in exits:
        return exits[move_key]
    else:
        return current


assert describe(descriptions, "cave") == "A damp cave."
assert move(exits, "cave", "east") == "forest"
assert move(exits, "cave", "north") == "cave"

## Milestone 3

### Explore the adventure

A fixed list of directions drives the loop. The hero collects each room's item once, the quit choice ends the journey, and one seeded event may cause damage without changing the route length.

In [ ]:
def apply_event(hero, roll):
    if roll <= 2:
        hero.take_damage(3)


hero = Hero("Ada")
current = "cave"
scripted_moves = ["east", "west", "east", "east", "q"]
move_number = 0
final_move = ""
roll = random.randint(1, 6)
apply_event(hero, roll)

while move_number < len(scripted_moves):
    direction = scripted_moves[move_number]
    next_room = move(exits, current, direction)
    final_move = next_room
    move_number = move_number + 1

    if next_room == "quit":
        break

    current = next_room
    print(describe(descriptions, current))
    if current in room_items:
        item = room_items[current]
        if item not in hero.inventory:
            hero.pick_up(item)

event_hit = Hero("Hit test")
apply_event(event_hit, 1)
event_miss = Hero("Miss test")
apply_event(event_miss, 6)

assert roll == 2
assert hero.health == 17
assert len(hero.inventory) == 2
assert hero.inventory[0] == "sword"
assert event_hit.health == 17
assert event_miss.health == 20
assert final_move == "quit"

## Milestone 4

### Save and load the journey

The save file stores one value per line. A line counter rebuilds the hero's name, health, and inventory, then direct comparisons verify the round trip.

In [ ]:
with open("adventure_save.txt", "w") as save_file:
    save_file.write(hero.name + "\n")
    save_file.write(str(hero.health) + "\n")
    for item in hero.inventory:
        save_file.write(item + "\n")

loaded_name = ""
loaded_health = 0
loaded_items = []
line_number = 0

with open("adventure_save.txt", "r") as save_file:
    for saved_line in save_file.readlines():
        cleaned_line = saved_line.strip()
        if line_number == 0:
            loaded_name = cleaned_line
        elif line_number == 1:
            loaded_health = int(cleaned_line)
        else:
            loaded_items.append(cleaned_line)
        line_number = line_number + 1

assert loaded_name == hero.name
assert loaded_health == hero.health
assert len(loaded_items) == len(hero.inventory)
assert loaded_items[0] == hero.inventory[0]
assert loaded_items == hero.inventory